# Exploratory Data Analysis
## Student Academic Performance Predictor

This notebook explores the UCI Student Performance (Math class) dataset to understand feature distributions, correlations, and their relationship with the Pass/Fail outcome.

**Target**: `G3 >= 10` = **Pass (1)**, else **Fail (0)**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

# Load data
df = pd.read_csv('../data/student_data.csv')
df['Pass'] = (df['G3'] >= 10).astype(int)
df['Outcome'] = df['Pass'].map({1: 'Pass', 0: 'Fail'})

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# ----- 1. Class Distribution -----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['Outcome'].value_counts()
colors = ['#22c55e', '#ef4444']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution (Pass vs Fail)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Outcome'); axes[0].set_ylabel('Number of Students')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 13})
axes[1].set_title('Pass / Fail Split', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Class imbalance ratio (Pass:Fail): {counts["Pass"]/counts["Fail"]:.2f}:1')

In [ ]:
# ----- 2. Numeric Feature Correlation Heatmap -----
numeric_cols = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
                'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health',
                'absences', 'G1', 'G2', 'G3', 'Pass']
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ----- 3. Key Feature Distributions vs Outcome -----
key_features = {
    'studytime': 'Weekly Study Time (1-4)',
    'failures':  'Past Class Failures',
    'absences':  'School Absences',
    'Medu':      "Mother's Education (0-4)",
    'goout':     'Going Out Frequency (1-5)',
    'Walc':      'Weekend Alcohol (1-5)',
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (col, label) in enumerate(key_features.items()):
    pass_vals = df[df['Pass'] == 1][col]
    fail_vals = df[df['Pass'] == 0][col]
    axes[i].hist(pass_vals, bins=15, alpha=0.6, color='#22c55e', label='Pass', edgecolor='white')
    axes[i].hist(fail_vals, bins=15, alpha=0.6, color='#ef4444', label='Fail', edgecolor='white')
    axes[i].set_title(label, fontweight='bold')
    axes[i].set_xlabel(col); axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Feature Distributions: Pass vs Fail Students', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ----- 4. Boxplots: Key Numeric Features by Outcome -----
boxplot_cols = ['studytime', 'failures', 'absences', 'famrel', 'health', 'goout']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(boxplot_cols):
    sns.boxplot(data=df, x='Outcome', y=col, palette={'Pass': '#22c55e', 'Fail': '#ef4444'},
                ax=axes[i], width=0.5)
    axes[i].set_title(f'{col} by Outcome', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Boxplots: Comparing Pass vs Fail Students', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ----- 5. Grade Progression: G1, G2, G3 -----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
grade_cols = ['G1', 'G2', 'G3']
grade_labels = ['1st Period Grade', '2nd Period Grade', 'Final Grade (G3)']

for i, (col, label) in enumerate(zip(grade_cols, grade_labels)):
    sns.histplot(data=df, x=col, hue='Outcome', bins=21, multiple='stack',
                 palette={'Pass': '#22c55e', 'Fail': '#ef4444'}, ax=axes[i])
    axes[i].set_title(label, fontweight='bold')
    axes[i].set_xlabel('Grade (0-20)')
    axes[i].axvline(10, color='white', linestyle='--', linewidth=1.5, label='Pass threshold')

plt.suptitle('Grade Distributions Across Assessment Periods', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ----- 6. Categorical Feature Pass Rates -----
cat_cols = ['sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 'famsup',
            'paid', 'activities', 'higher', 'internet', 'romantic']

pass_rates = {}
for col in cat_cols:
    pr = df.groupby(col)['Pass'].mean().reset_index()
    pr.columns = ['Category', 'PassRate']
    pr['Feature'] = col
    pass_rates[col] = pr

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    data = pass_rates[col]
    bars = axes[i].bar(data['Category'], data['PassRate'] * 100,
                       color=['#22c55e' if v >= 0.5 else '#ef4444' for v in data['PassRate']],
                       edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylim(0, 100)
    axes[i].set_ylabel('Pass Rate (%)')
    axes[i].axhline(50, color='gray', linestyle='--', linewidth=1)
    for bar, val in zip(bars, data['PassRate']):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val*100:.0f}%', ha='center', fontsize=9, fontweight='bold')

# Hide unused subplots
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Pass Rate by Categorical Feature', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ----- 7. Summary Statistics -----
print('=== Summary Statistics ===')
print(df.groupby('Outcome')[['studytime', 'failures', 'absences', 'Medu', 'Fedu', 'famrel']]
      .mean().round(2).to_string())

print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found.')

print('\n=== Data Types ===')
print(df.dtypes.value_counts())